# Chapter 3: Clustering — Finding Groups Without Labels

### Why This Matters

Regression and classification are **supervised learning** — you have a known correct answer (revenue, churn/no-churn) for every historical row, and the model learns to predict it. **Clustering** is **unsupervised** — there's no "correct" group label at all; the algorithm discovers structure in the data on its own. This is exactly the right tool for customer segmentation, where the whole point is *discovering* natural groupings you didn't predefine, rather than predicting a label you already know.

### 3.1 K-Means Clustering

In [ ]:
np.random.seed(42)
seg = pd.DataFrame({
    "Annual_Spend": np.concatenate([
        np.random.normal(20, 5, 40),    # a "low spend" group
        np.random.normal(55, 8, 40),    # a "mid spend" group
        np.random.normal(95, 10, 40)    # a "high spend" group
    ]),
    "Frequency": np.concatenate([
        np.random.normal(4, 1.5, 40),
        np.random.normal(12, 2, 40),
        np.random.normal(22, 3, 40)
    ])
})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(seg)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
seg["Cluster"] = kmeans.fit_predict(X_scaled)

sns.scatterplot(data=seg, x="Annual_Spend", y="Frequency", hue="Cluster", palette="Set2")
plt.title("Customer Segments (K-Means)")
plt.show()

print(seg.groupby("Cluster")[["Annual_Spend", "Frequency"]].mean().round(1))

⚠️ **Scaling before clustering is not optional — it's essential.** K-Means groups points based on straight-line distance; if `Annual_Spend` ranges from 0 to 100,000 and `Frequency` ranges from 0 to 30, the distance calculation will be almost entirely dominated by `Annual_Spend`, and `Frequency` will barely influence the resulting clusters at all, regardless of how meaningful it actually is. `StandardScaler` (Chapter 4) puts every feature on a comparable scale (mean 0, standard deviation 1) before clustering, so each feature contributes fairly.

`n_clusters=3` must be chosen by you — K-Means doesn't discover the "right" number of groups on its own. In real work, you'd typically try several values and use the **elbow method** (plotting within-cluster variance against `k` and looking for where the improvement sharply levels off) or the **silhouette score** to pick a defensible number, alongside domain judgment about how many distinct customer segments actually make business sense to act on (three tiers of a loyalty program are actionable; twelve micro-segments usually aren't).

### 3.2 Hierarchical Clustering — When You Want a Dendrogram

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

sample = seg.sample(30, random_state=42)     # a smaller sample keeps the dendrogram readable
Z = linkage(scaler.fit_transform(sample[["Annual_Spend", "Frequency"]]), method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Customer Index")
plt.ylabel("Distance")
plt.show()

Unlike K-Means, hierarchical clustering doesn't require you to specify the number of clusters upfront — it builds a full tree (**dendrogram**) showing how every point successively merges with its nearest neighbor/group, from individual points at the bottom up to one giant cluster at the top. You then choose how many clusters you want *after the fact*, by "cutting" the tree at a chosen height — genuinely useful when you want to visually inspect the natural grouping structure before committing to a specific count, or when you suspect groups exist at multiple levels of granularity (e.g., broad segments that further split into sub-segments).

### Cheat Sheet — Clustering

| Task | Code |
|---|---|
| Scale first (always) | `StandardScaler().fit_transform(X)` |
| K-Means | `KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)` |
| Choosing k | Elbow method or silhouette score — never guess |
| Hierarchical | `linkage(X_scaled, method="ward")` then `dendrogram(Z)` |
| Profile the clusters | `df.groupby("Cluster")[cols].mean()` |

---